# IBM Applied Data Science Capstone
## Falcon 9 landing prediction - interactive launch-site mapping

**Learner:** Djessi Jorge  
**Completed:** 4 August 2026

Folium is used to locate launch sites, cluster mission outcomes and measure proximity to
selected transport and coastal features near CCAFS LC-40.

In [1]:
from math import atan2, cos, radians, sin, sqrt
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition

launches = pd.read_csv("spacex_launch_geo.csv")
site_coordinates = (
    launches[["Launch Site", "Lat", "Long"]]
    .drop_duplicates()
    .sort_values("Launch Site")
    .reset_index(drop=True)
)
site_coordinates

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [2]:
centre = [29.559684888503615, -95.0830971930759]
launch_map = folium.Map(location=centre, zoom_start=4, tiles="OpenStreetMap")

folium.Circle(
    location=centre, radius=1000, color="#44546A", fill=True,
    popup="NASA Johnson Space Center"
).add_to(launch_map)

for row in site_coordinates.itertuples(index=False):
    location = [row.Lat, row.Long]
    folium.Circle(location=location, radius=1000, color="#F28E2B", fill=True).add_to(launch_map)
    folium.Marker(location=location, popup=row[0]).add_to(launch_map)

print(f"Mapped {len(site_coordinates)} launch-site labels.")

Mapped 4 launch-site labels.


In [3]:
cluster = MarkerCluster(name="Launch outcomes").add_to(launch_map)
for row in launches.itertuples(index=False):
    successful = int(row[10]) == 1
    folium.Marker(
        location=[row[11], row[12]],
        popup=f"Flight {row[0]} | {'Success' if successful else 'Failure'}",
        icon=folium.Icon(color="green" if successful else "red", icon="info-sign"),
    ).add_to(cluster)

MousePosition(
    position="topright",
    separator=" | ",
    prefix="Lat / Long:",
    num_digits=5,
).add_to(launch_map)

In [4]:
def haversine_km(point_a, point_b):
    radius = 6373.0
    lat1, lon1 = map(radians, point_a)
    lat2, lon2 = map(radians, point_b)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * radius * atan2(sqrt(value), sqrt(1 - value))

launch_site = (28.562302, -80.577356)
proximity_points = {
    "Perimeter road": (28.561695, -80.57918),
    "Coastline": (28.56321, -80.5679),
    "Rail line": (28.57209, -80.58528),
}

distances = []
for label, point in proximity_points.items():
    distance = haversine_km(launch_site, point)
    distances.append({"Feature": label, "Distance_km": round(distance, 2)})
    folium.Marker(point, popup=f"{label}: {distance:.2f} km").add_to(launch_map)
    folium.PolyLine([launch_site, point], color="#4E79A7", weight=3).add_to(launch_map)

pd.DataFrame(distances)

,Feature,Distance_km
0,Perimeter road,0.19
1,Coastline,0.93
2,Rail line,1.34


In [5]:
launch_map.save("spacex_launch_site_map.html")
print("Saved interactive map: spacex_launch_site_map.html")
launch_map

Saved interactive map: spacex_launch_site_map.html


### Interpretation

All launch sites are coastal. The selected CCAFS LC-40 point is approximately 0.19 km from
a perimeter road, 0.93 km from the coastline and 1.34 km from a rail line. The coastal
buffer protects populated areas while nearby transport infrastructure supports operations.